# Ders 08 - Çoklu Ajan Tasarım Deseni


## Kurulum


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Neden Çok Temsilcili Sistemler?

Seyahat planlaması gibi gerçek dünya görevleri, lojistik, yerel bilgi, bütçeleme ve daha fazlası gibi birçok farklı uzmanlık gerektirir. Her şeyi tek başına yapmaya çalışan bir temsilci hızla kontrol edilemez hale gelir.

Çok temsilcili sistemler bunu **uzmanlaşma** ile çözer: her temsilci bir uzmanlık alanına odaklanır ve bir genelciye göre daha yüksek kaliteli sonuçlar üretir. Ayrıca **ölçeklenebilirliği** arttırır — mevcut iş akışını yeniden yazmadan yeni temsilciler (örneğin, bir uçuş uzmanı, bir restoran eleştirmeni) ekleyebilirsiniz. Temsilciler, birinden diğerine bağlamı aktararak yapılandırılmış bir boru hattı aracılığıyla bir araya gelir.


## Uzmanlaşmış Temsilciler Oluşturma


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Ardışık Bir İş Akışı Oluşturma

`WorkflowBuilder` ajanları yönlendirilmiş bir grafik halinde bağlamanızı sağlar. Burada basit iki adımlı bir boru hattı oluşturuyoruz: **TravelPlanner** rota taslağını hazırlar, ardından **TravelConcierge** onu gözden geçirir ve geliştirir.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Çalışma Akışına Daha Fazla Temsilci Ekleme

Çoklu temsilci modelinin en büyük avantajlarından biri, genişletmenin ne kadar kolay olduğudur. Aşağıda, planı yolcunun bütçesine karşı kontrol eden, maliyetlerin sınırı aşabilecek öğeleri işaretleyen ve tasarruf sağlayan alternatifler öneren **BudgetReviewer** adlı bir temsilci ekliyoruz. Çalışma akışı artık üç temsilciyi sırayla çalıştırıyor:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Özet

Bu derste şunları öğrendiniz:

1. **Uzman ajanlar oluşturma** — her biri odaklanmış bir role sahip (planlama, konsiyerj, bütçe incelemesi).
2. **Ajanları sıralı bir iş akışına bağlama** `WorkflowBuilder` ve `add_edge` kullanarak.
3. **Çok ajanlı boru hattından çıktıyı akıtma**, hangi ajanın konuştuğunu takip ederek.
4. **Bir iş akışını genişletme**, mevcut ajanları değiştirmeden zincire yeni ajanlar ekleyerek.

Çok ajanlı tasarım deseni, her ajanı basit tutarken, tek bir ajanın başaramayacağı kadar zengin ve kapsamlı olarak gözden geçirilmiş sonuçlar üretir.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalardan veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
